# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrahmanshaheen1/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Check Colab Secrets."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

APRIL_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse connected.")

Warehouse connected.


In [2]:
feature_frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS march_impressions,

            SUM(COALESCE(gsc_clicks, 0))
                AS march_clicks,

            ROUND(
                100.0 *
                SUM(COALESCE(gsc_clicks, 0)) /
                NULLIF(
                    SUM(COALESCE(gsc_impressions, 0)),
                    0
                ),
                4
            ) AS march_ctr_pct,

            AVG(
                CASE
                    WHEN gsc_impressions > 0
                         AND gsc_avg_position > 0
                    THEN gsc_avg_position
                END
            ) AS march_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0
                    THEN report_date
                END
            ) AS march_active_days

        FROM {MARCH_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING
            SUM(COALESCE(gsc_impressions, 0)) >= 100
    ),

    april_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS april_impressions,

            COUNT(DISTINCT report_date)
                AS april_observed_days

        FROM {APRIL_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.*,
        a.april_impressions,

        CASE
            WHEN a.april_impressions
                 < 0.80 * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_next_month_decline

    FROM march_features AS m

    INNER JOIN april_outcomes AS a
        USING (
            client_hash_id,
            content_hash_id
        )

    WHERE a.april_observed_days > 0
""").df()

FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days",
]

print(f"Rows: {len(feature_frame):,}")
print(f"Clients: {feature_frame['client_hash_id'].nunique():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441
Clients: 44


In [3]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        feature_frame,
        groups=feature_frame["client_hash_id"],
    )
)

train_df = feature_frame.iloc[train_idx].copy()
test_df = feature_frame.iloc[test_idx].copy()

model = Pipeline(
    steps=[
        (
            "missing_values",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

model.fit(
    train_df[FEATURES],
    train_df["is_next_month_decline"],
)

test_df["decline_score"] = model.predict_proba(
    test_df[FEATURES]
)[:, 1]

print(
    "Validated grouped-client model recreated."
)

print(
    f"Score range: "
    f"{test_df['decline_score'].min():.3f} "
    f"to {test_df['decline_score'].max():.3f}"
)

Validated grouped-client model recreated.
Score range: 0.000 to 0.635


In [6]:
action_queue = test_df.copy()

# --------------------------------------------------
# Compare CTR with pages in similar search positions
# --------------------------------------------------

action_queue["position_bucket"] = pd.cut(
    action_queue["march_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "positions_4_10",
        "positions_11_20",
        "positions_21_50",
        "positions_51_plus",
    ],
    include_lowest=True,
)

train_for_ctr = train_df.copy()

train_for_ctr["position_bucket"] = pd.cut(
    train_for_ctr["march_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "positions_4_10",
        "positions_11_20",
        "positions_21_50",
        "positions_51_plus",
    ],
    include_lowest=True,
)

bucket_ctr = (
    train_for_ctr
    .groupby(
        "position_bucket",
        observed=True
    )["march_ctr_pct"]
    .median()
)

action_queue["position_median_ctr"] = (
    action_queue["position_bucket"]
    .map(bucket_ctr)
    .astype(float)
)

action_queue["weak_ctr"] = (
    action_queue["march_ctr_pct"]
    < action_queue["position_median_ctr"]
)

# --------------------------------------------------
# Use the ACTUAL model-score distribution
# --------------------------------------------------

high_threshold = action_queue["decline_score"].quantile(0.90)
medium_threshold = action_queue["decline_score"].quantile(0.60)

print(f"High-priority threshold (top 10%): {high_threshold:.3f}")
print(f"Medium-priority threshold (top 40%): {medium_threshold:.3f}")

# --------------------------------------------------
# Human-readable action archetypes
# --------------------------------------------------

conditions = [
    # High model score + weak CTR while already visible
    (
        (action_queue["decline_score"] >= high_threshold)
        & action_queue["weak_ctr"]
        & (action_queue["march_avg_position"] <= 20)
    ),

    # High model score, but CTR is not clearly the main issue
    (
        action_queue["decline_score"] >= high_threshold
    ),

    # Elevated but not top-priority score
    (
        (action_queue["decline_score"] >= medium_threshold)
        & (action_queue["decline_score"] < high_threshold)
    ),
]

actions = [
    "REVIEW_CTR_AND_REFRESH",
    "REVIEW_CONTENT_AND_RANKING",
    "MONITOR_AND_REVIEW",
]

reasons = [
    "HIGH_SCORE_LOW_CTR_FOR_POSITION",
    "HIGH_SCORE_OTHER_SIGNALS",
    "ELEVATED_DECLINE_SCORE",
]

action_queue["action_label"] = np.select(
    conditions,
    actions,
    default="MONITOR",
)

action_queue["reason_code"] = np.select(
    conditions,
    reasons,
    default="LOWER_DECLINE_SCORE",
)

# --------------------------------------------------
# Rank pages
# --------------------------------------------------

action_queue = (
    action_queue
    .sort_values(
        "decline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

action_queue.insert(
    0,
    "priority_rank",
    np.arange(1, len(action_queue) + 1)
)

display(
    action_queue[
        [
            "priority_rank",
            "decline_score",
            "march_impressions",
            "march_clicks",
            "march_ctr_pct",
            "march_avg_position",
            "march_active_days",
            "action_label",
            "reason_code",
        ]
    ].head(20)
)

High-priority threshold (top 10%): 0.593
Medium-priority threshold (top 40%): 0.496


,priority_rank,decline_score,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_days,action_label,reason_code
0,1,0.634635,992.0,0.0,0.0,0.663352,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
1,2,0.632474,447.0,0.0,0.0,1.926054,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
2,3,0.631453,256.0,0.0,0.0,2.514371,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
3,4,0.631304,1133.0,0.0,0.0,2.496049,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
4,5,0.631197,645.0,0.0,0.0,2.611243,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
5,6,0.631129,1499.0,0.0,0.0,2.550616,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
6,7,0.631064,587.0,0.0,0.0,2.691523,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
7,8,0.631015,842.0,0.0,0.0,2.689654,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
8,9,0.630770,698.0,0.0,0.0,2.841735,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION
9,10,0.630682,551.0,0.0,0.0,2.907586,31,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION


### Ranked-action logic

The validated Logistic Regression model produces a continuous decline score for each page. I use that score for prioritization rather than treating an arbitrary probability such as 0.70 as a universal decision threshold.

The highest 10% of model scores form the high-priority review group. Pages between the 60th and 90th percentiles form an elevated-priority group, while lower-scoring pages remain in monitoring.

The action also considers whether a page has weak CTR compared with other pages in a similar search-position bucket.

| Archetype | Action | Reason code |
|---|---|---|
| High score + weak CTR + position 1–20 | `REVIEW_CTR_AND_REFRESH` | `HIGH_SCORE_LOW_CTR_FOR_POSITION` |
| High score without the CTR pattern | `REVIEW_CONTENT_AND_RANKING` | `HIGH_SCORE_OTHER_SIGNALS` |
| Elevated score | `MONITOR_AND_REVIEW` | `ELEVATED_DECLINE_SCORE` |
| Lower score | `MONITOR` | `LOWER_DECLINE_SCORE` |

These actions are decision-support labels. A high score does not prove that a page will decline, and the action label does not prove that a refresh will improve performance.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is designed for a content editor, SEO specialist, or content manager who has more pages than can be reviewed manually.

The validated model ranks pages by a measured decline score. The action layer then adds a readable reason code so a human can understand why a page was placed near the top of the queue.

The intended workflow is:

1. Use the ranked queue to decide which pages deserve attention first.
2. Read the reason code and supporting March metrics.
3. Review the actual page, search intent, snippet, content quality, and business importance.
4. Decide whether to refresh, improve click capture, investigate ranking issues, continue monitoring, or take no action.

The model is a **decision-support tool**. It prioritizes investigation; it does not make the final editorial decision.

### Decay and refresh insight

The FlyRank research paper observed that mature and stale content can lose performance and that refreshed older content can perform strongly. This supports including refresh review as one possible human action.

However, the current model does not contain an explicit content-age or freshness feature. Therefore, a high model score should not be interpreted as evidence that a page is stale. The reviewer must check freshness separately before deciding that a refresh is appropriate.

### Cost and value thinking

Not every flagged page has the same business value.

A false positive can waste editorial time or lead to an unnecessary change on a healthy page. A false negative can leave a genuinely declining page unattended.

Pages with stronger existing visibility may justify earlier human review because more search exposure may be at stake. However, impressions are context for prioritization and value; they do not prove that a particular content change will improve future performance.

### Limits

This playbook was evaluated using a March-to-April observation window and a grouped-client validation design. Its results should not be assumed to transfer unchanged to every client, topic, season, or future time period.

The model uses five search-performance features and cannot observe all reasons a page may gain or lose performance. Changes in search demand, competitors, SERP features, tracking quality, seasonality, site changes, and content context may affect outcomes without being represented in the model.

The scores and actions should therefore be interpreted as measured, directional decision-support signals rather than guaranteed predictions or causal recommendations.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every recommended page must be reviewed by a person before any content action is taken.

For a high-priority page, the reviewer should check:

1. Whether the March metrics are based on enough impressions and active days to be meaningful.
2. Whether zero or weak CTR may be caused by search intent, tracking problems, or SERP behavior rather than the page content itself.
3. Whether the title and meta description match the likely search intent.
4. Whether the page is still accurate, useful, and complete.
5. Whether the content is actually stale or outdated before choosing a refresh action.
6. Whether the page has strategic or business value that makes the work worth the editorial cost.
7. Whether the recommendation conflicts with other site changes, seasonality, migrations, or known campaigns.

### No-go list — what should NOT be automated

This playbook should **not** automatically:

* rewrite or publish page content;
* change titles or meta descriptions;
* delete, prune, redirect, or deindex pages;
* decide that a page is stale without checking content age and freshness;
* treat the model score as proof that a page will decline;
* assume that refreshing a page will improve search performance;
* make business-priority decisions without human context;
* act on pages with suspicious, incomplete, or low-quality measurement data.

The safest use is:

**model ranking → reason code → human review → human decision → measured follow-up**

The system is therefore a prioritization and decision-support layer, not an autonomous content-optimization system.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The playbook should be monitored because search behavior, client mix, content strategy, and model performance can change over time.

The current model should be reviewed or retrained when one or more of these conditions occurs:

* **Ranking quality drops:** Precision@50 falls materially below the validated grouped-client result of 0.700.
* **Feature distribution shifts:** March-style inputs such as CTR, clicks, average position, impressions, or active-day coverage move substantially from the ranges used to train the model.
* **Client mix changes:** The system is used on many new clients, industries, or content types that were not well represented in the training data.
* **Search environment changes:** Major search-engine updates, SERP changes, tracking changes, migrations, or measurement changes alter how the input signals behave.
* **Action mix looks wrong:** Human reviewers repeatedly reject the same reason code or action archetype, suggesting that the mapping no longer reflects useful editorial decisions.
* **Error patterns change:** High-scoring negatives or low-scoring positives become more common or show a new repeated pattern.
* **Enough new labeled data exists:** A new observation window is available with enough later outcomes to rebuild the model and compare it fairly with the existing version.

### Light monitoring plan

This is a non-production playbook, so monitoring can remain simple:

1. Track Precision@20 and Precision@50 on each new evaluation window.
2. Record how often reviewers accept, reject, or modify the suggested action.
3. Review the distribution of model scores and key features.
4. Inspect a small sample of the highest-ranked recommendations and the largest errors.
5. Retrain only when the new data provides enough evidence that the current model is no longer representative.

Retraining should not happen automatically on a schedule just because time has passed. It should be triggered by measured drift, weaker validation performance, new client populations, or meaningful changes in the underlying search environment.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --------------------------------------------------
# Export ranked queue
# --------------------------------------------------

export_columns = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "decline_score",
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days",
    "position_bucket",
    "position_median_ctr",
    "action_label",
    "reason_code",
]

queue_path = "work/outputs/action_playbook_queue.csv"

action_queue[export_columns].to_csv(
    queue_path,
    index=False,
)

print(f"Saved ranked queue: {queue_path}")


# --------------------------------------------------
# Export a metrics / configuration receipt
# --------------------------------------------------

playbook_receipt = {
    "selected_model": "logistic_regression",
    "validation_design": "grouped_by_client",
    "validated_precision_at_20": 0.850,
    "validated_precision_at_50": 0.700,
    "high_priority_definition": "top_10_percent_of_decline_scores",
    "high_priority_threshold": float(high_threshold),
    "elevated_priority_definition": "60th_to_90th_percentile_of_decline_scores",
    "medium_priority_threshold": float(medium_threshold),
    "features": FEATURES,
    "human_review_required": True,
    "automatic_content_changes_allowed": False,
}

receipt_path = "work/outputs/w07_action_playbook_metrics.json"

with open(receipt_path, "w") as f:
    json.dump(
        playbook_receipt,
        f,
        indent=2,
    )

print(f"Saved metrics receipt: {receipt_path}")
print(json.dumps(playbook_receipt, indent=2))

Saved ranked queue: work/outputs/action_playbook_queue.csv
Saved metrics receipt: work/outputs/w07_action_playbook_metrics.json
{
  "selected_model": "logistic_regression",
  "validation_design": "grouped_by_client",
  "validated_precision_at_20": 0.85,
  "validated_precision_at_50": 0.7,
  "high_priority_definition": "top_10_percent_of_decline_scores",
  "high_priority_threshold": 0.5928350987912301,
  "elevated_priority_definition": "60th_to_90th_percentile_of_decline_scores",
  "medium_priority_threshold": 0.49649025670206764,
  "features": [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days"
  ],
  "human_review_required": true,
  "automatic_content_changes_allowed": false
}


In [8]:
action_summary = (
    action_queue
    .groupby(
        ["action_label", "reason_code"],
        observed=True,
    )
    .size()
    .reset_index(name="page_count")
    .sort_values(
        "page_count",
        ascending=False,
    )
)

action_summary

,action_label,reason_code,page_count
0,MONITOR,LOWER_DECLINE_SCORE,5013
1,MONITOR_AND_REVIEW,ELEVATED_DECLINE_SCORE,2507
3,REVIEW_CTR_AND_REFRESH,HIGH_SCORE_LOW_CTR_FOR_POSITION,781
2,REVIEW_CONTENT_AND_RANKING,HIGH_SCORE_OTHER_SIGNALS,55


### Export summary

The notebook exports the ranked action queue to `work/outputs/action_playbook_queue.csv`. This file contains the model score, supporting March metrics, action label, and reason code for each ranked page.

The current action distribution is:

- 5,013 pages: `MONITOR`
- 2,507 pages: `MONITOR_AND_REVIEW`
- 781 pages: `REVIEW_CTR_AND_REFRESH`
- 55 pages: `REVIEW_CONTENT_AND_RANKING`

This distribution keeps the playbook selective. Most pages are not assigned an immediate intervention, while the highest-ranked pages receive stronger human-review recommendations.

The notebook also writes `work/outputs/w07_action_playbook_metrics.json`, which records the validated model, grouped-client validation design, Precision@20 and Precision@50 results, score thresholds, feature set, and the requirement for human review.

The exported queue is intended as an input to the research paper's recommendations section. It is not a production automation feed.

## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.